# Seurat Wrapper + Circos Visualization

In this notebook, you can learn how to visualize the output of a NicheNet analysis in a circos plot (chord diagram). This follows the same workflow as the wrapper notebook, then adds circos visualizations.

Note that we generally recommend combining heatmaps (ligand activity, ligand-target, ligand-receptor, expression, LFC) over circos plots, especially when many sender cell types are involved. Circos plots can be less informative and lead to wrong interpretation in complex cases.

We again use NICHE-seq data from Medaglia et al. (2017).

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import nichenetr as nn

### Load data and networks

In [ ]:
ligand_target_matrix = nn.load_ligand_target_matrix("mouse")
lr_network = nn.load_lr_network("mouse")
weighted_networks = nn.load_weighted_networks("mouse")

lr_network = lr_network[["from", "to"]].drop_duplicates()

adata = nn.load_seurat_obj()
adata = nn.alias_to_symbol_anndata(adata, "mouse")

## Perform the NicheNet analysis

We use the top 20 ligands to avoid overcrowding the circos plot.

In [ ]:
sender_celltypes = ["CD4 T", "Treg", "Mono", "NK", "B", "DC"]

nichenet_output = nn.nichenet_seuratobj_aggregate(
    receiver="CD8 T",
    adata=adata,
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    sender=sender_celltypes,
    celltype_col="celltype",
    ligand_target_matrix=ligand_target_matrix,
    lr_network=lr_network,
    weighted_networks=weighted_networks,
    top_n_ligands=20,
)

In [ ]:
nichenet_output["ligand_activities"]

In [ ]:
# Ligand-target heatmap
nichenet_output["ligand_target_heatmap"]
plt.show()

## Circos Plots to Visualize Ligand-Target and Ligand-Receptor Interactions

### Assign ligands to sender cells

We assign each top ligand to the sender cell type where it is most highly expressed.

In [ ]:
ligand_type_indication_df = nn.assign_ligands_to_celltype(
    adata,
    celltype_col="celltype",
    sender_celltypes=sender_celltypes,
    ligands_oi=nichenet_output["top_ligands"],
)

print(ligand_type_indication_df.head(10))
print("\nLigand type distribution:")
print(ligand_type_indication_df["ligand_type"].value_counts())

### Define the ligand-target links of interest

We show only links with a weight higher than a predefined cutoff (bottom 40% of scores removed).

In [ ]:
active_ligand_target_links_df = nichenet_output["ligand_target_df"].copy()
active_ligand_target_links_df["target_type"] = "LCMV-DE"

circos_links = nn.get_ligand_target_links_oi(
    ligand_type_indication_df,
    active_ligand_target_links_df,
    cutoff=0.40,
)

circos_links.head()

In [ ]:
# Define colors for cell types and target genes
ligand_colors = {
    "General": "#377EB8",
    "NK": "#4DAF4A",
    "B": "#984EA3",
    "Mono": "#FF7F00",
    "DC": "#FFFF33",
    "Treg": "#F781BF",
    "CD8 T": "#E41A1C",
}
target_colors = {"LCMV-DE": "#999999"}

vis_circos_obj = nn.prepare_circos_visualization(
    circos_links,
    ligand_colors=ligand_colors,
    target_colors=target_colors,
)

### Render circos plots

In [ ]:
# Circos plot without transparency
nn.make_circos_plot(vis_circos_obj, transparency=False, show=True)

In [ ]:
# Circos plot with transparency based on regulatory potential
nn.make_circos_plot(vis_circos_obj, transparency=True, show=True)

### Visualize ligand-receptor interactions as a circos plot

We create a ligand-receptor chord diagram using the weighted ligand-receptor dataframe.

In [ ]:
lr_network_top_df = nichenet_output["ligand_receptor_df"].copy()
lr_network_top_df["target_type"] = "LCMV_CD8T_receptor"
lr_network_top_df = lr_network_top_df.rename(columns={"receptor": "target"})
lr_network_top_df = lr_network_top_df.merge(ligand_type_indication_df, on="ligand")

receptor_colors = {"LCMV_CD8T_receptor": "#E41A1C"}

vis_circos_receptor_obj = nn.prepare_circos_visualization(
    lr_network_top_df,
    ligand_colors=ligand_colors,
    target_colors=receptor_colors,
)

In [ ]:
nn.make_circos_plot(
    vis_circos_receptor_obj,
    transparency=False,
    link_visible=True,
    show=True,
)